In [1]:
"""
XGB-only Stat-AQ ablation study for the ovarian-cancer pipeline
===============================================================

Purpose
-------
This script is a focused ablation version of the supplied
"Main_Code Final(2).ipynb". It keeps the same primary dataset,
80:20 raw train/held-out split, leakage-safe binned preprocessing,
10-fold outer / 5-fold inner nested cross-validation, ANN settings,
QUBO settings, XGB hyperparameter search space, random seed, and
classification metrics.

It intentionally DOES NOT rerun the full Stat-AQ + XGB pipeline,
because that result already exists in the main study.

Only these three ablation conditions are run:

1) no_statistical_filtering
   Preprocessing -> ANN saliency + QUBO -> strict union -> XGB
   (Kruskal-Wallis and Cramer's V are both skipped.)

2) no_ann_saliency
   Preprocessing -> statistical filtering -> QUBO only -> XGB

3) no_qubo_optimization
   Preprocessing -> statistical filtering -> ANN saliency only -> XGB

XGB is held fixed across all three conditions. This isolates the
contribution of upstream Stat-AQ components rather than mixing feature-
selection ablations with classifier changes.

Outputs
-------
For each ablation condition, the script saves only:
- 10 outer-fold nested-CV results;
- nested-CV mean +/- SD summary;
- one final 20% held-out test result;
- optional held-out bootstrap 95% CIs.

No plots, calibration analysis, XAI, clinical baselines, representation
sensitivity, alternative classifiers, or full Stat-AQ rerun are included.

Run in Colab/local
------------------
    python ovarian_stat_aq_ablation_xgb.py

If Kaggle download is unavailable, set DATA_FILE below to the local
"Supplementary data 1.xlsx" path.
"""

from __future__ import annotations

import json
import os
import random
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------
# Configuration copied from the supplied main pipeline
# ---------------------------------------------------------------------

INSTALL_MISSING_PACKAGES = True
RANDOM_STATE = 42

DATASET_HANDLE = "saurabhshahane/predict-ovarian-cancer"
DATA_FILE: Optional[str] = None
OUTPUT_DIR = "ovarian_stat_aq_ablation_xgb_outputs"
CLEAN_OUTPUT_DIR = True

TARGET_COL = "TYPE"
TEST_SIZE = 0.20
OUTER_SPLITS = 10
INNER_SPLITS = 5
N_ITER_SEARCH = 30

RUN_BOOTSTRAP_CI = True
BOOTSTRAP_N = 2000
BOOTSTRAP_SEED = 42
BOOTSTRAP_METRIC_CI_LEVEL = 0.95

DROP_COLS_PRE_SPECIFIED = ["SUBJECT_ID", "NEU", "CA72-4"]
BINARY_COLS = ["TYPE", "Menopause"]

LOG_BIN_COLS = [
    "AFP", "ALP", "ALT", "AST", "CA125", "CA19-9", "CEA",
    "DBIL", "IBIL", "TBIL", "CREA", "GGT", "UA", "HE4",
]

QUANTILE_BIN_COLS = [
    "EO#", "EO%", "MONO#", "MONO%", "PCT", "PDW", "PLT", "BASO#", "BASO%",
]

EQUAL_BIN_COLS = [
    "AG", "Age", "ALB", "BUN", "Ca", "CL", "CO2CP", "GLO", "GLU.", "HCT", "HGB",
    "K", "LYM#", "LYM%", "MCH", "MCV", "Mg", "MPV", "Na", "PHOS", "RBC", "RDW", "TP",
]

CLINICALLY_PROTECTED_FEATURES = ["CA125", "HE4", "Menopause", "AFP", "ALB", "DBIL"]

KRUSKAL_ALPHA = 0.05
CRAMERS_V_THRESHOLD = 0.90

ANN_EPOCHS = 10
ANN_BATCH_SIZE = 32
ANN_POSITIVE_IMPORTANCE_ONLY = True
ANN_MIN_FEATURES = 5
ANN_PERMUTATION_REPEATS = 10

QUBO_ALPHA = 1.0
QUBO_BETA = 0.5
QUBO_LAMBDA = 0.2
QUBO_CARDINALITY_GAMMA = 0.0
QUBO_TARGET_K: Optional[int] = None
QUBO_NUM_READS = 100
REQUIRE_OPENJIJ_SQA = True

ABLATION_CONDITIONS = [
    "no_statistical_filtering",
    "no_ann_saliency",
    "no_qubo_optimization",
]


# ---------------------------------------------------------------------
# Environment and data loading
# ---------------------------------------------------------------------

def ensure_packages() -> None:
    """Install only packages needed by this focused ablation script."""
    if not INSTALL_MISSING_PACKAGES:
        return

    required = {
        "kagglehub": "kagglehub",
        "openpyxl": "openpyxl",
        "scipy": "scipy",
        "sklearn": "scikit-learn",
        "pandas": "pandas",
        "numpy": "numpy",
        "xgboost": "xgboost",
        "tensorflow": "tensorflow",
        "openjij": "openjij",
    }

    missing: List[str] = []
    for import_name, package_name in required.items():
        try:
            __import__(import_name)
        except Exception:
            missing.append(package_name)

    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet"] + missing
        )


def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except Exception:
        pass


def make_output_dir() -> Path:
    out = Path(OUTPUT_DIR)
    if CLEAN_OUTPUT_DIR and out.exists():
        import shutil
        shutil.rmtree(out)
    out.mkdir(parents=True, exist_ok=True)
    return out


def find_dataset_file(download_root: str) -> str:
    candidates: List[str] = []
    for root, _, files in os.walk(download_root):
        for fn in files:
            lower = fn.lower()
            if lower.endswith((".xlsx", ".xls", ".csv")):
                candidates.append(os.path.join(root, fn))

    if not candidates:
        raise FileNotFoundError(f"No CSV/XLSX file found under {download_root}")

    preferred = []
    for p in candidates:
        name = os.path.basename(p).lower()
        if "supplementary" in name and "data" in name and ("1" in name or "one" in name):
            preferred.append(p)

    if preferred:
        return sorted(preferred)[0]
    return sorted(candidates)[0]


def load_raw_dataset() -> Tuple[pd.DataFrame, str]:
    if DATA_FILE and os.path.exists(DATA_FILE):
        data_path = DATA_FILE
    else:
        try:
            import kagglehub
            downloaded_path = kagglehub.dataset_download(DATASET_HANDLE)
            data_path = find_dataset_file(downloaded_path)
        except Exception as exc:
            raise RuntimeError(
                "Could not download/read the Kaggle dataset. Set DATA_FILE manually "
                "to the local Supplementary data 1.xlsx path. Original error: "
                + str(exc)
            ) from exc

    if data_path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(data_path)
    elif data_path.lower().endswith(".csv"):
        df = pd.read_csv(data_path)
    else:
        raise ValueError(f"Unsupported data file: {data_path}")

    df.columns = [str(c).strip() for c in df.columns]
    return df, data_path


def standardize_target(y: Any) -> pd.Series:
    from sklearn.preprocessing import LabelEncoder

    y_series = pd.Series(y).copy()
    if y_series.dtype == "object" or str(y_series.dtype).startswith("category"):
        encoded = LabelEncoder().fit_transform(y_series)
        return pd.Series(encoded, index=y_series.index, name=TARGET_COL)
    return y_series.astype(int)


def split_raw_holdout(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    from sklearn.model_selection import train_test_split

    if TARGET_COL not in df.columns:
        raise ValueError(
            f"Target column {TARGET_COL!r} not found. Columns: {list(df.columns)}"
        )

    y = standardize_target(df[TARGET_COL])
    X = df.drop(columns=[TARGET_COL])

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    return (
        X_train_raw.reset_index(drop=True),
        X_test_raw.reset_index(drop=True),
        y_train.reset_index(drop=True),
        y_test.reset_index(drop=True),
    )


# ---------------------------------------------------------------------
# Leakage-safe preprocessing copied from the main pipeline
# ---------------------------------------------------------------------

@dataclass
class ParseAudit:
    column: str
    total_values: int
    missing_before: int
    parsed_missing_after: int
    comma_values: int
    quoted_or_tabbed_values: int
    greater_than_values: int
    less_than_values: int
    non_numeric_to_missing: int


def parse_numeric_with_audit(
    series: Any,
    column: str,
) -> Tuple[pd.Series, ParseAudit]:
    s = pd.Series(series).copy()
    missing_before = int(s.isna().sum())
    raw = s.astype("string")

    comma_values = int(raw.fillna("").str.contains(",", regex=False).sum())
    quoted_or_tabbed_values = int(
        raw.fillna("").str.contains(r"[\"'\t]", regex=True).sum()
    )
    greater_than_values = int(
        raw.fillna("").str.contains(r"^\s*>\s*=?", regex=True).sum()
    )
    less_than_values = int(
        raw.fillna("").str.contains(r"^\s*<\s*=?", regex=True).sum()
    )

    cleaned = raw.str.strip()
    cleaned = cleaned.str.replace("\t", "", regex=False)
    cleaned = cleaned.str.replace("\"", "", regex=False)
    cleaned = cleaned.str.replace("'", "", regex=False)
    cleaned = cleaned.str.replace(",", "", regex=False)
    cleaned = cleaned.str.replace(r"^\s*[<>]=?\s*", "", regex=True)
    cleaned = cleaned.str.replace(r"[^0-9eE+\-.]", "", regex=True)
    parsed = pd.to_numeric(cleaned, errors="coerce")

    parsed_missing_after = int(parsed.isna().sum())
    non_numeric_to_missing = max(0, parsed_missing_after - missing_before)

    audit = ParseAudit(
        column=column,
        total_values=len(series),
        missing_before=missing_before,
        parsed_missing_after=parsed_missing_after,
        comma_values=comma_values,
        quoted_or_tabbed_values=quoted_or_tabbed_values,
        greater_than_values=greater_than_values,
        less_than_values=less_than_values,
        non_numeric_to_missing=non_numeric_to_missing,
    )
    return parsed, audit


def make_equal_edges(values: Any, bins: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0 or np.nanmin(arr) == np.nanmax(arr):
        return np.array([-np.inf, np.inf])

    edges = np.linspace(np.nanmin(arr), np.nanmax(arr), bins + 1)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return np.unique(edges)


def make_quantile_edges(values: Any, q: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0 or np.nanmin(arr) == np.nanmax(arr):
        return np.array([-np.inf, np.inf])

    quantiles = np.linspace(0, 1, q + 1)
    edges = np.quantile(arr, quantiles)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return np.unique(edges)


def apply_edges(values: Any, edges: Any) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if len(edges) <= 2:
        return np.zeros(len(arr), dtype=int)
    return np.digitize(arr, np.asarray(edges)[1:-1], right=False).astype(int)


class LeakSafePreprocessor:
    """Fit-on-training-only preprocessing reproduced from the main pipeline."""

    def __init__(self, representation: str = "binned"):
        if representation != "binned":
            raise ValueError("This ablation script uses only the paper's primary binned representation.")

        self.representation = representation
        self.columns_: List[str] = []
        self.scale100_cols_: List[str] = []
        self.impute_means_: Dict[str, float] = {}
        self.bin_edges_: Dict[str, Any] = {}
        self.parse_audits_: List[ParseAudit] = []
        self.fitted_ = False

    def _clean_base(self, X: Any, fit: bool = False) -> pd.DataFrame:
        Xc = pd.DataFrame(X).copy()

        drop_existing = [c for c in DROP_COLS_PRE_SPECIFIED if c in Xc.columns]
        if drop_existing:
            Xc = Xc.drop(columns=drop_existing)

        if TARGET_COL in Xc.columns:
            Xc = Xc.drop(columns=[TARGET_COL])

        if fit:
            self.columns_ = list(Xc.columns)
        else:
            missing_cols = [c for c in self.columns_ if c not in Xc.columns]
            if missing_cols:
                raise ValueError(f"Transform data is missing columns: {missing_cols}")
            Xc = Xc[self.columns_]

        parsed = pd.DataFrame(index=Xc.index)
        audits: List[ParseAudit] = []
        original_dtypes = Xc.dtypes.astype(str).to_dict()

        for col in Xc.columns:
            parsed_col, audit = parse_numeric_with_audit(Xc[col], col)
            parsed[col] = parsed_col
            audits.append(audit)

        if fit:
            self.parse_audits_ = audits
            self.scale100_cols_ = [
                c
                for c in Xc.columns
                if ("float" in original_dtypes[c].lower())
                or ("object" in original_dtypes[c].lower())
            ]

            self.impute_means_ = {}
            for col in parsed.columns:
                mean_val = parsed[col].mean()
                if not np.isfinite(mean_val):
                    mean_val = 0.0
                self.impute_means_[col] = float(mean_val)

        for col in parsed.columns:
            parsed[col] = parsed[col].fillna(self.impute_means_.get(col, 0.0))

        for col in self.scale100_cols_:
            if col in parsed.columns:
                parsed[col] = np.ceil(parsed[col].astype(float) * 100.0).astype(int)

        for col in parsed.columns:
            if col not in self.scale100_cols_:
                parsed[col] = np.ceil(parsed[col].astype(float)).astype(int)

        return parsed

    def fit(self, X: Any, y: Optional[Any] = None) -> "LeakSafePreprocessor":
        base = self._clean_base(X, fit=True)

        for col in base.columns:
            if col in BINARY_COLS:
                continue
            if col in LOG_BIN_COLS:
                vals = np.log1p(base[col].clip(lower=0).astype(float))
                self.bin_edges_[col] = make_equal_edges(vals, bins=5)
            elif col in QUANTILE_BIN_COLS:
                self.bin_edges_[col] = make_quantile_edges(base[col], q=4)
            elif col in EQUAL_BIN_COLS:
                self.bin_edges_[col] = make_equal_edges(base[col], bins=5)
            else:
                self.bin_edges_[col] = make_equal_edges(base[col], bins=5)

        self.fitted_ = True
        return self

    def transform(self, X: Any) -> pd.DataFrame:
        if not self.fitted_ and not self.columns_:
            raise RuntimeError("Preprocessor must be fitted before transform.")

        base = self._clean_base(X, fit=False)
        out = pd.DataFrame(index=base.index)

        for col in base.columns:
            if col in BINARY_COLS:
                out[col] = base[col].astype(int)
            elif col in LOG_BIN_COLS:
                vals = np.log1p(base[col].clip(lower=0).astype(float))
                out[col] = apply_edges(vals, self.bin_edges_[col])
            else:
                out[col] = apply_edges(base[col], self.bin_edges_[col])

        return out.astype(int)

    def fit_transform(self, X: Any, y: Optional[Any] = None) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)


# ---------------------------------------------------------------------
# Statistical filtering copied from the main pipeline
# ---------------------------------------------------------------------

def kruskal_filter(
    X: pd.DataFrame,
    y: pd.Series,
    alpha: float = KRUSKAL_ALPHA,
) -> Tuple[List[str], pd.DataFrame]:
    from scipy.stats import kruskal

    selected: List[str] = []
    rows: List[Dict[str, Any]] = []

    y_arr = pd.Series(y).reset_index(drop=True)
    Xr = X.reset_index(drop=True)

    for feature in Xr.columns:
        g0 = Xr.loc[y_arr == 0, feature]
        g1 = Xr.loc[y_arr == 1, feature]

        if g0.nunique(dropna=True) < 2 and g1.nunique(dropna=True) < 2:
            stat, p_value = np.nan, 1.0
        else:
            try:
                stat, p_value = kruskal(g0, g1)
            except Exception:
                stat, p_value = np.nan, 1.0

        keep = bool(p_value < alpha)
        rows.append(
            {
                "feature": feature,
                "kruskal_H": stat,
                "p_value": p_value,
                "keep": keep,
            }
        )
        if keep:
            selected.append(feature)

    if not selected:
        # Same safety fallback as the supplied main code.
        tmp = pd.DataFrame(rows).sort_values("p_value")
        selected = tmp.head(min(10, len(tmp)))["feature"].tolist()
        tmp.loc[tmp["feature"].isin(selected), "keep"] = True
        rows = tmp.to_dict("records")

    return selected, pd.DataFrame(rows).sort_values("p_value")


def cramers_v(x: Any, y: Any) -> float:
    from scipy.stats import chi2_contingency

    if pd.Series(x).nunique(dropna=True) <= 1 or pd.Series(y).nunique(dropna=True) <= 1:
        return 0.0

    table = pd.crosstab(x, y)
    if table.shape[0] <= 1 or table.shape[1] <= 1:
        return 0.0

    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    if n == 0:
        return 0.0

    phi2 = chi2 / n
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom <= 0:
        return 0.0

    return float(np.sqrt(phi2 / denom))


def cramers_v_matrix(X: pd.DataFrame) -> pd.DataFrame:
    cols = list(X.columns)
    mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for c1 in cols:
        for c2 in cols:
            mat.loc[c1, c2] = cramers_v(X[c1], X[c2])
    return mat


def cramers_v_filter(
    X: pd.DataFrame,
    threshold: float = CRAMERS_V_THRESHOLD,
    protected: Sequence[str] = CLINICALLY_PROTECTED_FEATURES,
) -> Tuple[List[str], List[str], pd.DataFrame]:
    mat = cramers_v_matrix(X)
    binary_cols = [
        c
        for c in X.columns
        if set(X[c].dropna().unique()).issubset({0, 1})
    ]

    upper = mat.where(np.triu(np.ones(mat.shape), k=1).astype(bool))

    dropped: List[str] = []
    for col in upper.columns:
        if col in protected or col in binary_cols:
            continue
        high_corr = upper[col].dropna()
        if any(high_corr > threshold):
            dropped.append(col)

    kept = [c for c in X.columns if c not in dropped]
    return kept, dropped, mat


def apply_statistical_filtering(
    X: pd.DataFrame,
    y: pd.Series,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Kruskal-Wallis followed by Cramer's V, fitted on active training data only."""
    kw_features, kw_df = kruskal_filter(X, y)
    X_kw = X[kw_features]
    kept, dropped, _ = cramers_v_filter(X_kw)
    X_filtered = X_kw[kept]

    details = {
        "kruskal_retained": kw_features,
        "cramers_dropped": dropped,
        "candidate_features": kept,
    }
    return X_filtered, details


# ---------------------------------------------------------------------
# ANN saliency copied from the main pipeline, with plotting removed
# ---------------------------------------------------------------------

class KerasWrapper:
    def __init__(self, model: Any):
        self.model = model

    def fit(self, X: Any, y: Any) -> "KerasWrapper":
        return self

    def predict(self, X: Any) -> np.ndarray:
        return (
            self.model.predict(X, verbose=0).reshape(-1) > 0.5
        ).astype(int)

    def predict_proba(self, X: Any) -> np.ndarray:
        p = self.model.predict(X, verbose=0).reshape(-1, 1)
        return np.hstack([1 - p, p])


def ann_permutation_features(
    X: pd.DataFrame,
    y: pd.Series,
) -> Tuple[List[str], pd.DataFrame]:
    from sklearn.inspection import permutation_importance
    from sklearn.model_selection import train_test_split
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler

    if X.shape[1] == 0:
        return [], pd.DataFrame(columns=["feature", "ann_importance"])

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        stratify=y,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    try:
        import tensorflow as tf
    except Exception:
        tf = None

    if tf is not None:
        set_seed(RANDOM_STATE)
        model = tf.keras.Sequential(
            [
                tf.keras.layers.Input(shape=(X.shape[1],)),
                tf.keras.layers.Dense(64, activation="relu"),
                tf.keras.layers.Dense(1, activation="sigmoid"),
            ]
        )
        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        model.fit(
            X_train_s,
            y_train,
            epochs=ANN_EPOCHS,
            batch_size=ANN_BATCH_SIZE,
            verbose=0,
        )
        wrapped = KerasWrapper(model)
        perm = permutation_importance(
            wrapped,
            X_val_s,
            y_val,
            n_repeats=ANN_PERMUTATION_REPEATS,
            random_state=RANDOM_STATE,
            scoring="accuracy",
        )
        importances = perm.importances_mean
    else:
        # This is the same fallback present in the supplied main code.
        mlp = MLPClassifier(
            hidden_layer_sizes=(64,),
            max_iter=500,
            random_state=RANDOM_STATE,
        )
        mlp.fit(X_train_s, y_train)
        perm = permutation_importance(
            mlp,
            X_val_s,
            y_val,
            n_repeats=ANN_PERMUTATION_REPEATS,
            random_state=RANDOM_STATE,
            scoring="accuracy",
        )
        importances = perm.importances_mean

    ranking = pd.DataFrame(
        {
            "feature": X.columns,
            "ann_importance": importances,
        }
    ).sort_values("ann_importance", ascending=False)

    if ANN_POSITIVE_IMPORTANCE_ONLY:
        selected = ranking.loc[
            ranking["ann_importance"] > 0,
            "feature",
        ].tolist()
    else:
        selected = ranking.head(
            max(ANN_MIN_FEATURES, min(11, len(ranking)))
        )["feature"].tolist()

    if len(selected) < ANN_MIN_FEATURES:
        selected = ranking.head(
            min(ANN_MIN_FEATURES, len(ranking))
        )["feature"].tolist()

    return selected, ranking


# ---------------------------------------------------------------------
# QUBO component copied from the main pipeline, with plotting removed
# ---------------------------------------------------------------------

def build_qubo(
    X: pd.DataFrame,
    y: pd.Series,
) -> Tuple[
    Dict[Tuple[int, int], float],
    np.ndarray,
    np.ndarray,
    Dict[str, Any],
]:
    n = X.shape[1]
    y_arr = pd.Series(y).astype(float).values

    relevance: List[float] = []
    for col in X.columns:
        x = X[col].astype(float).values
        if np.std(x) == 0 or np.std(y_arr) == 0:
            relevance.append(0.0)
        else:
            val = np.corrcoef(x, y_arr)[0, 1]
            if not np.isfinite(val):
                val = 0.0
            relevance.append(abs(float(val)))

    relevance_arr = np.asarray(relevance)
    redundancy = np.abs(
        pd.DataFrame(X).corr().fillna(0.0).values
    )

    qubo: Dict[Tuple[int, int], float] = {}

    for i in range(n):
        qubo[(i, i)] = (
            -QUBO_ALPHA * relevance_arr[i]
            + QUBO_LAMBDA
        )

    for i in range(n):
        for j in range(i + 1, n):
            qubo[(i, j)] = (
                QUBO_BETA * redundancy[i, j]
            )

    if QUBO_CARDINALITY_GAMMA > 0 and QUBO_TARGET_K is not None:
        for i in range(n):
            qubo[(i, i)] = (
                qubo.get((i, i), 0.0)
                + QUBO_CARDINALITY_GAMMA
                * (1 - 2 * QUBO_TARGET_K)
            )
        for i in range(n):
            for j in range(i + 1, n):
                qubo[(i, j)] = (
                    qubo.get((i, j), 0.0)
                    + 2 * QUBO_CARDINALITY_GAMMA
                )

    config = {
        "objective": (
            "min E(z)=sum_i(-alpha*relevance_i+lambda)*z_i "
            "+ sum_i<j beta*redundancy_ij*z_i*z_j "
            "+ optional gamma*(sum z-k)^2"
        ),
        "alpha_relevance_reward": QUBO_ALPHA,
        "beta_redundancy_penalty": QUBO_BETA,
        "lambda_sparsity_penalty": QUBO_LAMBDA,
        "gamma_cardinality_penalty": QUBO_CARDINALITY_GAMMA,
        "target_k": QUBO_TARGET_K,
        "num_reads": QUBO_NUM_READS,
        "random_state": RANDOM_STATE,
        "required_solver": "openjij.SQASampler_simulated_quantum_annealing",
        "no_classical_fallback": bool(REQUIRE_OPENJIJ_SQA),
        "relevance_definition": (
            "absolute Pearson/point-biserial correlation "
            "with target on active training data"
        ),
        "redundancy_definition": (
            "absolute Pearson correlation between candidate "
            "predictors on active training data"
        ),
    }

    return qubo, relevance_arr, redundancy, config


def solve_qubo(
    qubo: Dict[Tuple[int, int], float],
) -> Tuple[Dict[int, int], str, float]:
    try:
        import openjij as oj
    except Exception as exc:
        raise RuntimeError(
            "OpenJij is required for the QUBO ablation conditions. "
            "Install it with: pip install openjij. "
            "No classical fallback is used."
        ) from exc

    sampler = oj.SQASampler()
    try:
        sampleset = sampler.sample_qubo(
            Q=qubo,
            num_reads=QUBO_NUM_READS,
        )
    except Exception as exc:
        raise RuntimeError(
            "OpenJij SQASampler failed. No classical fallback is used."
        ) from exc

    first = sampleset.first
    sample = {
        int(k): int(v)
        for k, v in first.sample.items()
    }
    energy = float(first.energy)

    return (
        sample,
        "openjij.SQASampler_simulated_quantum_annealing",
        energy,
    )


def qubo_features(
    X: pd.DataFrame,
    y: pd.Series,
) -> Tuple[List[str], Dict[str, Any]]:
    qubo, relevance, _, config = build_qubo(X, y)
    sample, sampler_name, energy = solve_qubo(qubo)

    config["sampler"] = sampler_name
    config["best_energy"] = energy
    config["strict_qubo_output_only"] = True

    selected = [
        X.columns[i]
        for i in range(X.shape[1])
        if int(sample.get(i, 0)) == 1
    ]

    return selected, config


# ---------------------------------------------------------------------
# Ablation feature selection
# ---------------------------------------------------------------------

def select_ablation_features(
    X_preprocessed: pd.DataFrame,
    y: pd.Series,
    condition: str,
) -> Tuple[List[str], Dict[str, Any]]:
    """
    Return the selected feature list for exactly one ablation condition.

    No fallback features are added to QUBO-only output because doing so would
    change the ablation being measured.
    """
    if condition not in ABLATION_CONDITIONS:
        raise ValueError(
            f"Unknown ablation condition: {condition}. "
            f"Expected one of {ABLATION_CONDITIONS}"
        )

    y = pd.Series(y).reset_index(drop=True)
    X_preprocessed = X_preprocessed.reset_index(drop=True)

    if condition == "no_statistical_filtering":
        X_candidates = X_preprocessed.copy()
        stat_details = {
            "statistical_filtering_used": False,
            "candidate_features": list(X_candidates.columns),
        }

        ann_features, _ = ann_permutation_features(
            X_candidates,
            y,
        )
        q_features, q_config = qubo_features(
            X_candidates,
            y,
        )

        selected = sorted(
            set(ann_features) | set(q_features)
        )
        details = {
            "condition": condition,
            **stat_details,
            "ann_used": True,
            "qubo_used": True,
            "ann_features": ann_features,
            "qubo_features": q_features,
            "combine_rule": "strict_union",
            "qubo_config": q_config,
        }

    elif condition == "no_ann_saliency":
        X_candidates, stat_details = apply_statistical_filtering(
            X_preprocessed,
            y,
        )

        q_features, q_config = qubo_features(
            X_candidates,
            y,
        )
        selected = list(q_features)

        details = {
            "condition": condition,
            "statistical_filtering_used": True,
            **stat_details,
            "ann_used": False,
            "qubo_used": True,
            "ann_features": [],
            "qubo_features": q_features,
            "combine_rule": "QUBO_only",
            "qubo_config": q_config,
        }

    else:  # no_qubo_optimization
        X_candidates, stat_details = apply_statistical_filtering(
            X_preprocessed,
            y,
        )

        ann_features, _ = ann_permutation_features(
            X_candidates,
            y,
        )
        selected = list(ann_features)

        details = {
            "condition": condition,
            "statistical_filtering_used": True,
            **stat_details,
            "ann_used": True,
            "qubo_used": False,
            "ann_features": ann_features,
            "qubo_features": [],
            "combine_rule": "ANN_only",
        }

    if not selected:
        raise RuntimeError(
            f"Ablation condition {condition!r} produced zero features. "
            "The script stops instead of adding a fallback that would alter "
            "the intended ablation."
        )

    return selected, details


# ---------------------------------------------------------------------
# XGB only: exact search space from the supplied main pipeline
# ---------------------------------------------------------------------

def xgb_model_space() -> Dict[str, Any]:
    try:
        from xgboost import XGBClassifier
    except Exception as exc:
        raise RuntimeError(
            "XGBoost is required. Install it with: pip install xgboost"
        ) from exc

    return {
        "estimator": XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": [100, 200, 300, 400, 500],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "max_depth": [3, 5, 7, 9, 11],
            "min_child_weight": [1, 3, 5, 7],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
            "gamma": [0, 0.1, 0.25, 0.5],
            "reg_alpha": [0.0, 0.1, 1.0, 5.0],
            "reg_lambda": [0.0, 0.1, 1.0, 5.0],
        },
    }


def tune_xgb(
    X: pd.DataFrame,
    y: pd.Series,
):
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

    spec = xgb_model_space()
    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        n_iter=N_ITER_SEARCH,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
        random_state=RANDOM_STATE,
        error_score=np.nan,
        refit=True,
    )

    search.fit(X, y)
    return search


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

def proba_positive(model: Any, X: Any) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if p.ndim == 2 and p.shape[1] > 1:
            return p[:, 1]
        return p.reshape(-1)

    if hasattr(model, "decision_function"):
        z = model.decision_function(X)
        return 1.0 / (1.0 + np.exp(-z))

    return model.predict(X).astype(float)


def metrics_from_predictions(
    y_true: Any,
    y_pred: Any,
    y_prob: Any,
) -> Dict[str, Any]:
    from sklearn.metrics import (
        accuracy_score,
        brier_score_loss,
        confusion_matrix,
        f1_score,
        log_loss,
        roc_auc_score,
    )

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.clip(
        np.asarray(y_prob).astype(float),
        1e-7,
        1 - 1e-7,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )
    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )
    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )
    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    try:
        ll = log_loss(
            y_true,
            y_prob,
            labels=[0, 1],
        )
    except Exception:
        ll = np.nan

    try:
        brier = brier_score_loss(
            y_true,
            y_prob,
        )
    except Exception:
        brier = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity_Recall": sensitivity,
        "Specificity": specificity,
        "PPV_Precision": ppv,
        "NPV": npv,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": auc,
        "Log_Loss": ll,
        "Brier": brier,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def bootstrap_metric_ci(
    y_true: Any,
    y_pred: Any,
    y_prob: Any,
    n_boot: int = BOOTSTRAP_N,
) -> pd.DataFrame:
    rng = np.random.default_rng(BOOTSTRAP_SEED)

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    n = len(y_true)
    rows: List[Dict[str, Any]] = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue

        rows.append(
            metrics_from_predictions(
                y_true[idx],
                y_pred[idx],
                y_prob[idx],
            )
        )

    boot = pd.DataFrame(rows)

    alpha = 1.0 - float(BOOTSTRAP_METRIC_CI_LEVEL)
    low_q = 100.0 * alpha / 2.0
    high_q = 100.0 * (1.0 - alpha / 2.0)

    ci_rows: List[Dict[str, Any]] = []
    metrics = [
        "Accuracy",
        "Sensitivity_Recall",
        "Specificity",
        "PPV_Precision",
        "NPV",
        "F1",
        "ROC_AUC",
        "Log_Loss",
        "Brier",
    ]

    for metric in metrics:
        vals = (
            boot[metric].dropna().values
            if metric in boot
            else []
        )

        if len(vals) == 0:
            low = high = np.nan
        else:
            low = float(np.percentile(vals, low_q))
            high = float(np.percentile(vals, high_q))

        ci_rows.append(
            {
                "Metric": metric,
                "CI_level": BOOTSTRAP_METRIC_CI_LEVEL,
                "CI_low": low,
                "CI_high": high,
            }
        )

    return pd.DataFrame(ci_rows)


# ---------------------------------------------------------------------
# Nested CV and final held-out evaluation
# ---------------------------------------------------------------------

NESTED_METRIC_COLUMNS = [
    "Accuracy",
    "Sensitivity_Recall",
    "Specificity",
    "PPV_Precision",
    "NPV",
    "F1",
    "ROC_AUC",
    "Log_Loss",
    "Brier",
]


def nested_cv_ablation(
    condition: str,
    X_train_raw: pd.DataFrame,
    y_train: pd.Series,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    from sklearn.model_selection import StratifiedKFold

    outer_cv = StratifiedKFold(
        n_splits=OUTER_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows: List[Dict[str, Any]] = []

    for fold, (tr_idx, val_idx) in enumerate(
        outer_cv.split(X_train_raw, y_train),
        start=1,
    ):
        print(
            f"\n[{condition}] outer fold "
            f"{fold}/{OUTER_SPLITS}"
        )

        X_tr_raw = X_train_raw.iloc[tr_idx].reset_index(drop=True)
        X_val_raw = X_train_raw.iloc[val_idx].reset_index(drop=True)
        y_tr = y_train.iloc[tr_idx].reset_index(drop=True)
        y_val = y_train.iloc[val_idx].reset_index(drop=True)

        prep = LeakSafePreprocessor(representation="binned")
        X_tr_prep = prep.fit_transform(
            X_tr_raw,
            y_tr,
        ).reset_index(drop=True)

        X_val_prep = prep.transform(
            X_val_raw
        ).reset_index(drop=True)

        selected_features, _ = select_ablation_features(
            X_tr_prep,
            y_tr,
            condition,
        )

        X_tr_sel = X_tr_prep[selected_features]
        X_val_sel = X_val_prep[selected_features]

        search = tune_xgb(
            X_tr_sel,
            y_tr,
        )
        best_model = search.best_estimator_

        p_val = proba_positive(
            best_model,
            X_val_sel,
        )
        pred_val = (
            p_val >= 0.5
        ).astype(int)

        metrics = metrics_from_predictions(
            y_val,
            pred_val,
            p_val,
        )

        row = {
            "Ablation": condition,
            "Fold": fold,
            "Model": "XGB",
            "Num_Selected_Features": len(selected_features),
            "Selected_Features": ";".join(selected_features),
            "Best_Params": json.dumps(
                search.best_params_,
                default=str,
            ),
            **metrics,
        }
        rows.append(row)

        print(
            "  features="
            f"{len(selected_features):2d} | "
            f"ACC={metrics['Accuracy']:.4f} | "
            f"AUC={metrics['ROC_AUC']:.4f} | "
            f"F1={metrics['F1']:.4f}"
        )

    fold_df = pd.DataFrame(rows)

    summary: Dict[str, Any] = {
        "Ablation": condition,
        "Model": "XGB",
        "Outer_Folds": OUTER_SPLITS,
    }

    for metric in NESTED_METRIC_COLUMNS:
        vals = pd.to_numeric(
            fold_df[metric],
            errors="coerce",
        )
        summary[f"{metric}_mean"] = vals.mean()
        summary[f"{metric}_std"] = vals.std()

    summary_df = pd.DataFrame([summary])
    return fold_df, summary_df


def final_heldout_ablation(
    condition: str,
    X_train_raw: pd.DataFrame,
    y_train: pd.Series,
    X_test_raw: pd.DataFrame,
    y_test: pd.Series,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    prep = LeakSafePreprocessor(representation="binned")
    X_train_prep = prep.fit_transform(
        X_train_raw,
        y_train,
    ).reset_index(drop=True)

    X_test_prep = prep.transform(
        X_test_raw
    ).reset_index(drop=True)

    selected_features, _ = select_ablation_features(
        X_train_prep,
        y_train,
        condition,
    )

    X_train_sel = X_train_prep[selected_features]
    X_test_sel = X_test_prep[selected_features]

    search = tune_xgb(
        X_train_sel,
        y_train,
    )
    model = search.best_estimator_

    p_test = proba_positive(
        model,
        X_test_sel,
    )
    pred_test = (
        p_test >= 0.5
    ).astype(int)

    metrics = metrics_from_predictions(
        y_test,
        pred_test,
        p_test,
    )

    heldout_df = pd.DataFrame(
        [
            {
                "Ablation": condition,
                "Model": "XGB",
                "Heldout_N": len(y_test),
                "Num_Selected_Features": len(selected_features),
                "Selected_Features": ";".join(selected_features),
                "Best_Params": json.dumps(
                    search.best_params_,
                    default=str,
                ),
                **metrics,
            }
        ]
    )

    bootstrap_df: Optional[pd.DataFrame] = None

    if RUN_BOOTSTRAP_CI:
        bootstrap_df = bootstrap_metric_ci(
            y_test,
            pred_test,
            p_test,
        )
        bootstrap_df.insert(
            0,
            "Ablation",
            condition,
        )
        bootstrap_df.insert(
            1,
            "Model",
            "XGB",
        )

    print(
        f"\n[{condition}] HELD-OUT | "
        f"features={len(selected_features)} | "
        f"ACC={metrics['Accuracy']:.4f} | "
        f"Sens={metrics['Sensitivity_Recall']:.4f} | "
        f"Spec={metrics['Specificity']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"AUC={metrics['ROC_AUC']:.4f}"
    )

    return heldout_df, bootstrap_df


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main() -> None:
    set_seed(RANDOM_STATE)
    ensure_packages()
    out_dir = make_output_dir()

    df_raw, source_path = load_raw_dataset()

    print("\nXGB-ONLY STAT-AQ ABLATION STUDY")
    print("=" * 72)
    print("Dataset:", source_path)
    print("Raw shape:", df_raw.shape)
    print("Conditions:", ", ".join(ABLATION_CONDITIONS))
    print(
        "Full Stat-AQ + XGB is intentionally NOT rerun; "
        "use the original paper result as the reference row."
    )

    X_train_raw, X_test_raw, y_train, y_test = split_raw_holdout(
        df_raw
    )

    print(
        f"\nRaw split: train={len(y_train)}, "
        f"held-out={len(y_test)}"
    )
    print(
        "Train classes:",
        y_train.value_counts().sort_index().to_dict(),
    )
    print(
        "Held-out classes:",
        y_test.value_counts().sort_index().to_dict(),
    )

    all_nested_folds: List[pd.DataFrame] = []
    all_nested_summaries: List[pd.DataFrame] = []
    all_heldout: List[pd.DataFrame] = []
    all_bootstrap: List[pd.DataFrame] = []

    for condition in ABLATION_CONDITIONS:
        print("\n" + "#" * 72)
        print("RUNNING:", condition)
        print("#" * 72)

        fold_df, nested_summary_df = nested_cv_ablation(
            condition,
            X_train_raw,
            y_train,
        )

        heldout_df, bootstrap_df = final_heldout_ablation(
            condition,
            X_train_raw,
            y_train,
            X_test_raw,
            y_test,
        )

        fold_df.to_csv(
            out_dir / f"{condition}_nested_cv_10fold_results.csv",
            index=False,
        )
        nested_summary_df.to_csv(
            out_dir / f"{condition}_nested_cv_summary.csv",
            index=False,
        )
        heldout_df.to_csv(
            out_dir / f"{condition}_heldout_result.csv",
            index=False,
        )

        if bootstrap_df is not None:
            bootstrap_df.to_csv(
                out_dir / f"{condition}_heldout_bootstrap_95ci.csv",
                index=False,
            )
            all_bootstrap.append(bootstrap_df)

        all_nested_folds.append(fold_df)
        all_nested_summaries.append(nested_summary_df)
        all_heldout.append(heldout_df)

    combined_nested_folds = pd.concat(
        all_nested_folds,
        ignore_index=True,
    )
    combined_nested_summary = pd.concat(
        all_nested_summaries,
        ignore_index=True,
    )
    combined_heldout = pd.concat(
        all_heldout,
        ignore_index=True,
    )

    combined_nested_folds.to_csv(
        out_dir / "ABLATION_ALL_nested_cv_10fold_results.csv",
        index=False,
    )
    combined_nested_summary.to_csv(
        out_dir / "ABLATION_ALL_nested_cv_summary.csv",
        index=False,
    )
    combined_heldout.to_csv(
        out_dir / "ABLATION_ALL_heldout_results.csv",
        index=False,
    )

    if all_bootstrap:
        pd.concat(
            all_bootstrap,
            ignore_index=True,
        ).to_csv(
            out_dir / "ABLATION_ALL_heldout_bootstrap_95ci.csv",
            index=False,
        )

    print("\n" + "=" * 72)
    print("NESTED-CV MEAN +/- SD")
    print("=" * 72)
    print(
        combined_nested_summary.to_string(
            index=False
        )
    )

    print("\n" + "=" * 72)
    print("HELD-OUT RESULTS")
    print("=" * 72)
    print(
        combined_heldout.to_string(
            index=False
        )
    )

    print("\nDONE.")
    print("Outputs:", out_dir.resolve())


if __name__ == "__main__":
    main()


Installing missing packages: ['openjij']


100%|██████████| 315k/315k [00:00<00:00, 59.9MB/s]

Extracting files...



XGB-ONLY STAT-AQ ABLATION STUDY
Dataset: /root/.cache/kagglehub/datasets/saurabhshahane/predict-ovarian-cancer/versions/1/Supplementary data 1.xlsx
Raw shape: (349, 51)
Conditions: no_statistical_filtering, no_ann_saliency, no_qubo_optimization
Full Stat-AQ + XGB is intentionally NOT rerun; use the original paper result as the reference row.

Raw split: train=279, held-out=70
Train classes: {0: 137, 1: 142}
Held-out classes: {0: 34, 1: 36}

########################################################################
RUNNING: no_statistical_filtering
########################################################################

[no_statistical_filtering] outer fold 1/10
  features=39 | ACC=0.8929 | AUC=0.9286 | F1=0.9032

[no_statistical_filtering] outer fold 2/10
  features=12 | ACC=0.8214 | AUC=0.7143 | F1=0.8485

[no_statistical_filtering] outer fold 3/10
  features=28 | ACC=0.8571 | AUC=0.9490 | F1=0.8667

[no_statistical_filtering] outer fold 4/10
  features=27 | ACC=0.8214 | AUC=0.8571 | 